## Project
Using Time Series Analysis To Forecast Future Vacancy Levels 

## Hypothesis
If we examine historical vacancy data with data science techniques,

we can identify key patterns that occur over time and contribute to accurate forecasting, 

to drive actionable insights on labour market tightness for monetary policy makers. 

Why this matters? 
For example: higher vacancy volumes -> lower unemployment -> higher inflation -> interest rate measures 

## Task 
with time estimates (h)

1. Automate scraping of files (0.5)
2. Consolidate data (0.5)
3. Plot data to highlight patterns and revisions (0.5)
4. Build a simple model to forecast vacancy levels (1)
5. Comment on evaluation, recommendations, and next steps. (0.5)

All code, including any helper modules, has been included in this notebook to keep it readable for the purpose of this time assessed task.  

# 1. Import libraries and data

o Automate the download (or scraping) of the CSV files on the ONS website with a subset of at least 20 

In [ ]:
# Import libraries

from __future__ import annotations
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone
import requests
import time
import random

import csv
import re
import pandas as pd
from dateutil.parser import parse as dtparse


In [ ]:
# Scrape data from the web

PREV_URL = "https://www.ons.gov.uk/employmentandlabourmarket/peopleinwork/employmentandemployeetypes/timeseries/ap2y/lms/previous"
RAW_DIR  = (Path.cwd() / "../data/raw").resolve()
RAW_DIR.mkdir(parents=True, exist_ok=True)
USER_AGENT = "FVL-Interview-Task/1.0 (+https://github.com/<seleklekterek>/fvl-project; contact:<197108180+seleklekterek@users.noreply.github.com>)"

def scrape_ap2y_csv_links(page_url: str) -> list[str]:
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT})
    r = s.get(page_url, timeout=15)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    urls = [
        urljoin(page_url, a["href"])
        for a in soup.select("a[href]")
        if "format=csv" in a["href"].lower()
    ]
    seen, out = set(), []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out

def _head_last_modified(s: requests.Session, url: str) -> datetime | None:
    try:
        h = s.head(url, timeout=15, allow_redirects=True)
        if h.status_code >= 400 or "Last-Modified" not in h.headers:
            g = s.get(url, stream=True, timeout=30, allow_redirects=True)
            g.raise_for_status()
            lm = g.headers.get("Last-Modified")
            g.close()
        else:
            lm = h.headers.get("Last-Modified")

        if not lm:
            return None
        dt = parsedate_to_datetime(lm)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        else:
            dt = dt.astimezone(timezone.utc)
        return dt
    except Exception:
        return None

def _make_dst_name(dt: datetime | None, ix: int) -> str:
    if dt:
        stamp = dt.strftime("%Y%m%d")
        return f"AP2Y_prev_{stamp}.csv"
    else:
        return f"AP2Y_prev_idx{ix:03d}.csv"

def download_csvs_chrono(
    urls: list[str],
    raw_dir: Path,
    limit: int = 20,
    delay_s: float = 0.15,
    max_retries: int = 6,
    backoff_base: float = 2.0,
    order: str = "asc",  
):
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT})
    raw_dir.mkdir(parents=True, exist_ok=True)

    meta = []
    for u in urls:
        lm = _head_last_modified(s, u)
        meta.append((u, lm))

    meta.sort(key=lambda x: (x[1] or datetime.max.replace(tzinfo=timezone.utc)))
    if order == "desc":
        meta.reverse()

    meta = meta[:limit]

    seen_names = set()
    saved = []
    for ix, (url, dt) in enumerate(meta, start=1):
        base = _make_dst_name(dt, ix)
        name = base
        k = 1
        while name in seen_names or (raw_dir / name).exists():
            k += 1
            stem, ext = base.rsplit(".", 1)
            name = f"{stem}__v{k}.{ext}"
        seen_names.add(name)

        dst = raw_dir / name

        if dst.exists() and dst.stat().st_size > 0:
            saved.append(dst)
            continue

        for attempt in range(max_retries):
            try:
                resp = s.get(url, timeout=15, allow_redirects=True)
                if resp.status_code == 429:
                    ra = resp.headers.get("Retry-After")
                    wait = float(ra) + random.uniform(0.2, 0.8) if ra else (backoff_base ** attempt) + random.uniform(0.2, 0.8)
                    time.sleep(wait)
                    continue
                resp.raise_for_status()
                dst.write_bytes(resp.content)
                if dst.stat().st_size > 0:
                    saved.append(dst)
                    time.sleep(delay_s + random.uniform(0, 0.5))
                break
            except requests.RequestException as exc:
                wait = (backoff_base ** attempt) + random.uniform(0.2, 0.8)
                time.sleep(wait)
                if attempt == max_retries - 1:
                    print(f"[warn] {url} -> {exc}")
            except Exception as exc:
                print(f"[warn] {url} -> {exc}")
                break
    return saved

# run
urls = scrape_ap2y_csv_links(PREV_URL)
downloaded = download_csvs_chrono(urls, RAW_DIR, limit=20, order="asc")  
len(downloaded), [p.name for p in downloaded[:5]]

Challenges - 
1. Couldn't download as many as 20 files in one go - had to amend the code to respect ONS rate limits 
2. Upon closer inspection a couple of files from 2019 appeared in the set - had to amend the code to get the first 20 in historical order

Next step -
1. Optimise speed of retrieval - at over a minute this is too slow (this is because of one extra network round trip per URL to learn dates before donwloading for correct ordering)

# 2. Prepare data

o Clean and consolidate downloaded csvs capturing observation and release dates for each value. 


In [ ]:
# Consolidate all monthly data

RAW_DIR = (Path.cwd() / "../data/raw").resolve()
PROCESSED_DIR = (Path.cwd() / "../data/processed").resolve()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MONTH_MAP = {"JAN":1,"FEB":2,"MAR":3,"APR":4,"MAY":5,"JUN":6,
             "JUL":7,"AUG":8,"SEP":9,"OCT":10,"NOV":11,"DEC":12}

# helpers
def _detect_header_and_data_start(csv_path: Path) -> tuple[dict[str, str], int, bool]:
    header: dict[str, str] = {}
    data_start = 0
    has_header_row = False

    with csv_path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.reader(f))

    def _is_numeric(x: str) -> bool:
        try:
            float(x.replace(",", ""))
            return True
        except Exception:
            return False

    period_pat = re.compile(
        r"""^
            \d{4}(
                \s+(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC) |
                \s+Q[1-4] |
                \s+M(?:[1-9]|1[0-2])
            )?
        $""",
        re.IGNORECASE | re.VERBOSE,
    )

    for i, row in enumerate(rows):
        if not row:
            continue
        first = (row[0] or "").strip().lstrip("\ufeff").lower()

        if first in {"date", "period", "time"}:
            data_start = i
            has_header_row = True
            break

        if len(row) >= 2:
            tok = (row[0] or "").strip()
            val = (row[1] or "").strip()
            if period_pat.match(tok) and _is_numeric(val):
                data_start = i
                has_header_row = False
                break

    for r in rows[:data_start]:
        if len(r) >= 2 and r[0] and r[1]:
            header[str(r[0]).strip().lower()] = str(r[1]).strip()
        elif len(r) == 1 and r[0]:
            txt = str(r[0])
            if ":" in txt:
                k, v = txt.split(":", 1)
                k = k.strip().lower()
                v = v.strip()
                if k:
                    header[k] = v

    return header, data_start, has_header_row

def _extract_vintage_date(header: dict[str, str], fp: Path) -> pd.Timestamp:
    val = header.get("release date")
    if val:
        try:
            return pd.Timestamp(dtparse(val, dayfirst=True).date()) # to ensure it defaults to UK format
        except Exception:
            pass
    head_text = ""
    try:
        with fp.open("r", encoding="utf-8", errors="ignore") as f:
            for _ in range(60):
                try:
                    head_text += next(f)
                except StopIteration:
                    break
    except Exception:
        head_text = ""

    m = re.search(r"release\s*date\s*[:|,]\s*([0-9]{4}-[0-9]{2}-[0-9]{2}|[0-9]{1,2}\s+\w+\s+[0-9]{4}|[0-9]{1,2}[\/\-\.][0-9]{1,2}[\/\-\.][0-9]{4})",
              head_text, flags=re.IGNORECASE)
    if m:
        try:
            return pd.Timestamp(dtparse(m.group(1), dayfirst=True).date())
        except Exception:
            pass

    m2 = re.search(r"(\d{4})(\d{2})(\d{2})", fp.name)
    if m2:
        y, mm, dd = map(int, m2.groups())
        return pd.Timestamp(y, mm, dd)

    return pd.Timestamp(fp.stat().st_mtime, unit="s").normalize()

def _parse_month_token(tok: str) -> pd.Timestamp | None:
    s = tok.strip().upper().replace("-", " ").replace("/", " ")
    parts = s.split()
    if len(parts) == 2:
        a, b = parts
        if a.isdigit() and len(a) == 4 and b in MONTH_MAP:
            return pd.Timestamp(int(a), MONTH_MAP[b], 1)
        if b.isdigit() and len(b) == 4 and a[:3] in MONTH_MAP:
            return pd.Timestamp(int(b), MONTH_MAP[a[:3]], 1)
    try:
        d = dtparse(tok, yearfirst=True, dayfirst=False).date()
        return pd.Timestamp(d.year, d.month, 1)
    except Exception:
        return None

def parse_single_csv(csv_path: Path) -> pd.DataFrame:
    header, data_start, has_header = _detect_header_and_data_start(csv_path)

    if has_header:
        df = pd.read_csv(csv_path, skiprows=data_start)
        cols_lower = {c.lower(): c for c in df.columns}
        date_col = cols_lower.get("date") or cols_lower.get("time") or cols_lower.get("period") or df.columns[0]
        value_col = (
            cols_lower.get("value")
            or cols_lower.get("values")
            or cols_lower.get("observation")
            or (df.columns[1] if len(df.columns) > 1 else df.columns[0])
        )
    else:
        df = pd.read_csv(csv_path, skiprows=data_start, header=None, names=["period", "value"])
        date_col, value_col = "period", "value"

    df = df[[date_col, value_col]].rename(columns={date_col: "period", value_col: "value"})
    df["obs_date"] = df["period"].astype(str).map(_parse_month_token)

    df["value"] = (
        df["value"].astype(str).str.replace(",", "", regex=False).pipe(pd.to_numeric, errors="coerce")
    )

    df = (
        df.dropna(subset=["obs_date", "value"])
          .loc[:, ["obs_date", "value"]]
          .sort_values("obs_date")
          .reset_index(drop=True)
    )

    vintage_date = _extract_vintage_date(header, csv_path)
    df.insert(1, "vintage_date", vintage_date)
    return df[["obs_date", "vintage_date", "value"]]

def build_tidy_vintages(csv_files: list[Path]) -> pd.DataFrame:
    parts = [parse_single_csv(fp) for fp in csv_files]  
    return (
        pd.concat(parts, ignore_index=True)
          .sort_values(["obs_date", "vintage_date"])
          .drop_duplicates(subset=["obs_date", "vintage_date"])
          .reset_index(drop=True)
    )

# run consolidation 
files = sorted(RAW_DIR.glob("AP2Y_prev_*.csv"))  
tidy = build_tidy_vintages(files)

out_csv = PROCESSED_DIR / "vacancies_vintages.csv"
tidy.to_csv(out_csv, index=False)

print(f"Saved: {out_csv}")
print(f"Rows: {len(tidy):,} | Vintages: {tidy['vintage_date'].nunique()} | Obs months: {tidy['obs_date'].nunique()}")


In [ ]:
# Quick spot check 

print("\n=== Head (first 8 rows) ===")
print(tidy.head(8).to_string(index=False))

print("\n=== Tail (last 8 rows) ===")
print(tidy.tail(8).to_string(index=False))

print("\n=== Latest 10 rows (by vintage_date then obs_date, both desc) ===")
print(
    tidy.sort_values(["vintage_date", "obs_date"], ascending=[False, False])
        .head(10)
        .to_string(index=False)
)

print("\n=== Vintages summary (latest 5) — distinct obs months per vintage ===")
print(
    tidy.groupby("vintage_date")["obs_date"]
        .nunique()
        .sort_index(ascending=False)
        .head(5)
        .to_string()
)

mono_ok = (
    tidy.sort_values(["vintage_date", "obs_date"])
        .groupby("vintage_date")["obs_date"]
        .apply(lambda s: s.is_monotonic_increasing)
        .all()
)
print(f"\nPer-vintage obs_date monotonically increasing: {mono_ok}")

Challenges 
1. Vintage date was not falling back to download date - needed to amend original code to ensure the release date is captured correctly 
2. Some dates were pulled through US format - needed to amend the code to ensure dates were consolidated in correct format (e.x. as 12-08 instead of 08-12)

Note: 
- These functions have been written in a simple, fit for purpose way, and extensive cleaning and checks has been skipped as we are dealing with a small and consistent data set. 
- Additional failsafes should be added later to allow for naming convention and formatting changes over time. 
- Solutions to help with speed/size can also be added later (e.x. parquet) 

# 3. Plot data 

o Plot how the vacancy estimates for a given month have changed across different vintages. 

o Highlight any patterns or revisions that occur over time.

In [ ]:
# Visualise revisions



# 4. Forecast data 

o Build a simple model to forecast future vacancy levels

# 5. Evaluation and recommendations

o Outline in words how you would evaluate the forecast performance while thinking about
how the data can change per vintage.

o What next steps would you take to take this analysis to the next level?